# **🐳 MODULE 54 — DOCKER & DEPLOYMENT**

# **3. Containerize AI Models**

## **3.1 What Is Model Containerization?**

**Model containerization** is the process of packaging an AI/ML model and its complete inference environment into a Docker image so it can run consistently across different supported environments.

Instead of deploying only the trained model file, we package the components required to perform inference.

These typically include:

- Trained model
- Preprocessing logic
- Inference code
- API layer
- Runtime
- Dependencies
- Configuration

The resulting Docker image can be used to create a container that serves model predictions through an API.

---

## **3.2 Model Containerization Architecture**

The model-serving pipeline:

```text
Model
 +
Preprocessing
 +
API
 +
Dependencies
        ↓
   Docker Image
        ↓
   Container
        ↓
    REST API
```

**🧠 AI Model**

Trained weights and architecture

**⚙️ Preprocessing + Inference**

Input preparation and prediction logic

**🌐 API + Dependencies**

FastAPI, Python runtime and libraries

**📦 Docker Image**

Complete packaged inference environment

**🚀 Docker Container**

Running model-serving application

**REST API**

Clients send requests and receive predictions

---

## **3.3 Components Required for Containerizing an AI Model**

### **1. Model**

The trained model is responsible for generating predictions from input data.

Examples:

- DistilBERT → Sentiment analysis
- YOLO → Object detection
- ResNet → Image classification
- Whisper → Speech recognition

The model may be stored as:

- Model weights inside the image
- Downloaded during image building
- Downloaded during container startup
- Mounted externally using storage or volumes

### **2. Preprocessing**

Preprocessing converts raw user input into the format expected by the model.

**For NLP:**

```text
Raw Text
    ↓
Tokenization
    ↓
Attention Mask
    ↓
Tensor Conversion
    ↓
Model Input
```

**For Computer Vision:**

```text
Raw Image
    ↓
Image Decoding
    ↓
Resize / Normalize
    ↓
Tensor Conversion
    ↓
Model Input
```

Preprocessing must be consistent with the preprocessing used when training or fine-tuning the model.

### **3. API Layer**

The API exposes model functionality to external applications.

For example, using FastAPI:

```http
POST /predict
```

A client sends input:

```json
{
  "text": "I love this product!"
}
```

The API:

1. Receives the request.
2. Validates the input.
3. Runs preprocessing.
4. Performs model inference.
5. Returns the prediction.

Example response:

```json
{
  "label": "POSITIVE",
  "confidence": 0.98
}
```

### **4. Dependencies**

The application requires a runtime and libraries to operate.

For an AI model-serving application:

```text
Python
├── FastAPI
├── Uvicorn
├── PyTorch
├── Transformers
├── Pillow
└── Ultralytics
```

These dependencies are defined in a dependency file, such as:

```text
requirements.txt
```

Docker installs the required dependencies when building the image.

---

## **3.4 How Docker Packages the Model-Serving Application**

Docker uses a **Dockerfile** to describe the steps required to create the application image.

Example:

```dockerfile
FROM python:3.10

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### **Build Process**

```text
Dockerfile
    ↓
Base Runtime
    ↓
Install Dependencies
    ↓
Copy Application Files
    ↓
Configure Startup Command
    ↓
Build Docker Image
```

The image contains the application environment required to launch the API.

---

## **3.5 Model Loading Inside a Container**

Models can be made available to containers using different strategies.

### **Strategy A — Model Included in the Docker Image**

```text
Docker Image
├── Application
├── Dependencies
└── Model Weights
```

The model is packaged together with the application.

**Advantages**

- Self-contained deployment
- Model availability immediately after deployment
- Versioned application and model together

**Disadvantages**

- Larger Docker images
- Longer build and transfer times
- Updating the model requires rebuilding the image

### **Strategy B — Model Downloaded at Runtime**

```text
Docker Image
├── Application
└── Dependencies

Container Startup
        ↓
Download Model
        ↓
Load into Memory
        ↓
Inference API Ready
```

For example, Transformers may download a model from Hugging Face when the container starts, if it isn't already cached.

**Advantages**

- Smaller initial application image
- Model versions can be configured independently
- Easier to use externally hosted model repositories

**Disadvantages**

- Requires network access when downloading
- Startup can take longer
- External model availability can affect startup
- Model version management must be handled carefully

### **Strategy C — Model Stored Externally and Mounted into the Container**

```text
External Storage
       │
       ▼
Docker Volume / Mount
       │
       ▼
Container
├── Application
├── Dependencies
└── Access to Model Files
```

The model files remain outside the image and are made accessible to the container through mounted storage.

**Advantages**

- Model persistence
- Avoids rebuilding images for model file changes
- Can reduce duplicated model storage

**Disadvantages**

- Requires storage management
- Deployment configuration becomes more complex
- Model paths and permissions must be configured properly

---

## **3.6 Running the Container as a REST API**

Once the Docker image has been built, a container can run the model-serving application.

```text
Client Application
       │
       │ HTTP Request
       ▼
Docker Container
       │
       ▼
FastAPI Endpoint
       │
       ▼
Preprocessing
       │
       ▼
AI Model Inference
       │
       ▼
Prediction
       │
       ▼
HTTP Response
```

The client does not need to install PyTorch, Transformers, or the model itself.

It only needs to communicate with the exposed API.

Example client request:

```python
import requests

response = requests.post(
    "http://localhost:8000/predict",
    json={
        "text": "This product is excellent!"
    }
)

print(response.json())
```

The model runs inside the container, while the client interacts with it remotely through HTTP.

---

## **3.7 Benefits of Containerizing AI Models**

| Benefit | Explanation |
|---|---|
| **Reproducibility** | Consistent runtime and dependency environment |
| **Portability** | Deploy across supported systems |
| **Isolation** | Separate application runtime from host environment |
| **Scalability** | Run multiple container instances |
| **Versioning** | Version application images and model-serving releases |
| **Simplified deployment** | Reduce manual environment setup |
| **Integration** | Connect models with APIs, frontends and other services |
| **CI/CD compatibility** | Automate building, testing and deploying images |

---

## **3.8 Limitations & Considerations**

Containerizing an AI model doesn't automatically make the model faster or more efficient.

Important considerations:

- Model size can significantly increase image size.
- CPU and RAM requirements remain.
- GPU access must be configured separately.
- CUDA and driver compatibility matters for GPU inference.
- Model loading may increase container startup time.
- Containers require resource allocation and monitoring.
- A container alone doesn't provide automatic scaling or orchestration.

**GPU Note:** A normal Docker container does not automatically gain GPU access simply because the host machine has an NVIDIA GPU. GPU-enabled runtime/configuration and compatible drivers are needed.

---

## **3.9 Project 53 — Our Practical Application**

Our existing project:

**FastAPI + DistilBERT + YOLO + Streamlit**

**👤 Client**

Browser / Streamlit interface

**🌐 FastAPI Backend**

REST API • Request validation

**🧠 DistilBERT**

Sentiment inference

**👁️ YOLO**

Object detection

**🐳 Docker Container**

Runtime • Dependencies • Configuration

We will containerize this application so that the API can run without manually configuring Python, FastAPI, Transformers, PyTorch and other dependencies on every deployment machine.

---

## **🧠 Quick Revision**

> **Containerizing an AI model means packaging the model-serving application, preprocessing logic, API, runtime, and dependencies into a Docker image, which can be used to run an isolated container exposing the model through a REST API.**

### **Interview Question 🎯**

**How do you deploy a machine learning model using Docker?**

> I package the inference code, preprocessing pipeline, model-loading logic, API application, and required dependencies into a Docker image using a Dockerfile. I build the image and run a container with the required ports, configuration, storage, and hardware access. The model is then exposed through API endpoints, allowing client applications to send requests and receive predictions without directly accessing the model environment.

---

# **4. Dockerizing FastAPI**

- Creating a Dockerfile for FastAPI
- Managing Python dependencies
- Configuring the application startup command
- Binding the API to `0.0.0.0`
- Building the image
- Running the container
- Testing API endpoints inside Docker

---
